In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
def sample_normal_X(n, d, mean=0, scale=1, corr=0, Sigma=None):
    """
    Sample X with iid normal entries
    :param n:
    :param d:
    :param mean:
    :param scale:
    :param corr:
    :param Sigma:
    :return:
    """
    if Sigma is not None:
        if np.isscalar(mean):
            mean = np.repeat(mean, d)
        X = np.random.multivariate_normal(mean, Sigma, size=n)
    elif corr == 0:
        X = np.random.normal(mean, scale, size=(n, d))
    else:
        Sigma = np.zeros((d, d)) + corr
        np.fill_diagonal(Sigma, 1)
        if np.isscalar(mean):
            mean = np.repeat(mean, d)
        X = np.random.multivariate_normal(mean, Sigma, size=n)
    return X

def sample_block_cor_X(n, d, rho, n_blocks, mean=0):
    """
    Sample X from N(mean, Sigma) where Sigma is a block diagnoal covariance matrix
    :param n:
    :param d:
    :param rho:
    :param n_blocks:
    :param mean:
    :return:
    """
    Sigma = np.zeros((d, d))
    block_size = d // n_blocks
    if np.isscalar(rho):
        rho = np.repeat(rho, n_blocks)
    for i in range(n_blocks):
        start = i * block_size
        end = (i + 1) * block_size
        if i == (n_blocks - 1):
            end = d
        Sigma[start:end, start:end] = rho[i]
    np.fill_diagonal(Sigma, 1)
    X = sample_normal_X(n=n, d=d, mean=mean, Sigma=Sigma)
    return X

In [28]:
n = 50
d = 20
n_blocks = 5
rho = 0.9

X = sample_block_cor_X(n, d, rho, n_blocks)
beta = np.zeros(d)
beta[0] = 1
sigma = 0.1
y = X @ beta + np.random.randn(n) * sigma

In [29]:
rf_model = RandomForestRegressor(max_features=0.33)
rf_model.fit(X, y)
rf_model.feature_importances_

array([0.35902538, 0.1307362 , 0.14733191, 0.19618024, 0.0057458 ,
       0.0086166 , 0.00519265, 0.0040953 , 0.00687986, 0.01026979,
       0.01492869, 0.01043859, 0.00621266, 0.00899589, 0.0185516 ,
       0.01040921, 0.02470835, 0.01635872, 0.0088772 , 0.00644538])